# Optimización Bajo Incertidumbre (IIND 4125)

## Proyecto 2: Programación Estocástica

**Formato:** reporte ejecutivo + notebook reproducible + presentación en clase

## Motivación

En el Proyecto 1, cada grupo formuló un problema de optimización bajo incertidumbre y experimentó con esquemas básicos de solución a pequeña escala. En la práctica, la incertidumbre exige cientos o miles de escenarios para capturar adecuadamente el comportamiento estocástico, y especialmente las colas de la distribución. Dado que el *Equivalente Determinístico* crece en variables y restricciones con cada escenario adicional, este proyecto les pide abordar esta limitación, implementando:

1. **Descomposición de Benders** — para partir el problema en un *master* de primera etapa y sub-problemas de segunda etapa.
2. **Sample Average Approximation (SAA)** — para estimar el verdadero objetivo estocástico con muestreo.
3. **Métricas formales de decisión** — VSS, EVPI y CVaR para cuantificar el valor de modelar incertidumbre y riesgo.

El resultado será una cadena computacional completa: generar escenarios → descomponer → resolver → evaluar → comparar.

## 1. Punto de Partida: Proyecto 1

Cada grupo debe retomar el problema desarrollado en el Proyecto 1. A esta altura, deben re-evaluar si deben mejorar sus modelos de incertidumbre $W$ y $\hat{W}$ (Mediocristan y Extremistan), teniendo en cuenta que se busca poder generar escenarios numerosos y considerar variables aleatorias y de decisión tanto continuas como enteras.

- La formulación de dos etapas (decisiones de primera etapa + repercusión) debe ser coherente con lo entregado en el Proyecto 1, aunque se admiten variaciones para adaptarse a las implementaciones requeridas.
- Los generadores de escenarios ($W$ y $\hat{W}$) se reutilizan, pero ahora deben poder generar $N$ escenarios arbitrarios (no sólo el set fijo anterior).

## 2. Componente Algorítmico: Tres Estrategias de Solución

Para el mismo problema y el mismo conjunto de escenarios, implementen y comparen las tres estrategias siguientes:

### (1) Deterministic Equivalent (DE)

La formulación extensiva completa. Ya la tienen del Proyecto 1, pero ahora la usarán como punto de referencia de correctitud y como baseline de tiempo computacional.

### (2) Benders Decomposition — Naive (Loop Iterativo)

Implementen el loop clásico:

```
Repetir:
    1. Resolver Master → obtener x̂, z_hat
    2. Para cada escenario s: resolver Sub(s, x̂) → obtener Q_s(x̂)
    3. Si z_hat_s < Q_s(x̂): agregar corte de Benders al Master
    4. Actualizar cotas (LB, UB)
    5. Si LB ≥ UB − ε: parar
```

**Tipo de corte:** Dado que la primera etapa de los tres problemas (MCF, FLA, ND) incluye variables binarias, los cortes clásicos de optimalidad (duales) no son directamente aplicables. Deben usar cortes combinatorios.

### (3) Branch-and-Cut con Callbacks (Lazy Constraints)

Implementen la misma lógica de generación de cortes, pero dentro del árbol de Branch & Bound de Gurobi usando *callbacks*.

Pueden elegir una de dos variantes:

- **Multi-cut:** un corte por escenario (variable $z_s$ por cada $s$).
- **Single-cut:** un solo corte agregado (variable $\theta$ única).

### Entregable Algorítmico

| Métrica | DE | Naive Benders | Branch-and-Cut |
|---|---|---|---|
| Objetivo óptimo | ✓ | ✓ (verificar = DE) | ✓ (verificar = DE) |
| Tiempo de solución (s) | ✓ | ✓ | ✓ |
| Número de iteraciones / cortes | — | ✓ | ✓ |
| Gráfico de convergencia (LB vs UB) | — | ✓ | ✓ |

**Validación crítica:** Las tres estrategias deben dar el **mismo objetivo óptimo** (salvo tolerancia numérica). Si no coinciden, hay un bug.

## 3. Escalabilidad: SAA y el Efecto de $N$

El Proyecto 1 usaba un número fijo (y pequeño) de escenarios. Ahora deben estudiar cómo cambia la solución cuando $N$ crece.

### 3.1 Experimento de Estabilidad

Para $N \in \{10, 25, 50, 100, 200, 500\}$ (ajusten según la tratabilidad de su problema):

1. Generen $M \geq 20$ réplicas independientes de $N$ escenarios.
2. Para cada réplica, resuelvan el problema estocástico (usando la estrategia más eficiente de la Sección 2).
3. Reporten la distribución del objetivo óptimo a través de las réplicas.

**Gráfico esperado:** Boxplot, histograma, o violin plot del objetivo vs $N$. La varianza debe decrecer con $N$.

### 3.2 Método de Muestreo

Comparen al menos dos métodos de generación de escenarios:

- **Monte Carlo** (MC) — muestreo uniforme.
- **Latin Hypercube Sampling** (LHS) — mejor cobertura del espacio.

### 3.3 Time-to-Solution vs $N$

Midan el tiempo de solución de cada estrategia para distintos $N$ y grafiquen tiempo vs $N$. Este gráfico debe evidenciar cuándo el DE se vuelve inviable y Benders se vuelve competitivo.

## 4. Métricas de Decisión: VSS, EVPI y CVaR

### 4.1 VSS — Value of the Stochastic Solution

$$\text{VSS} = \text{EEV} - \text{RP}$$

Donde:
- **RP** = resultado óptimo del programa estocástico (recourse problem).
- **EEV** = costo esperado de usar la solución del problema con valores esperados, evaluada en los escenarios estocásticos.

### 4.2 EVPI — Expected Value of Perfect Information

$$\text{EVPI} = \text{RP} - \text{WS}$$

Donde **WS** = expected wait-and-see value (promedio de resolver cada escenario individualmente).

El EVPI mide cuánto se estaría dipuesto a pagar por un oráculo perfecto.

### 4.3 CVaR — Optimización Averso al Riesgo

Implementen una versión de su modelo que minimice el **CVaR** del costo de segunda etapa en lugar del valor esperado, usando la linealización de Rockafellar-Uryasev:

$$\min \;\; \text{costo}_1(x) + \eta + \frac{1}{\alpha} \sum_s p_s \, u_s$$

$$\text{s.a.} \quad u_s \geq Q_s(x) - \eta, \quad u_s \geq 0 \quad \forall s$$

Exploren al menos dos niveles de $\alpha$ (e.g., $\alpha = 0.10$ y $\alpha = 0.05$).

### Entregable de Métricas

| Métrica | Mediocristan | Extremistan |
|---|---|---|
| RP (objetivo estocástico) | | |
| EEV (costo de ignorar incertidumbre) | | |
| **VSS** | | |
| WS (wait-and-see) | | |
| **EVPI** | | |
| CVaR₉₀ de la solución neutral | | |
| CVaR₉₀ de la solución risk-averse | | |


## 5. Backtesting Extendido

Igual que en el Proyecto 1, evalúen las soluciones obtenidas *fuera de muestra*, generando un nuevo conjunto grande de realizaciones del "mundo" $W$ (no de $\hat{W}$), logrando más soluciones para comparar:

| Solución | Descripción |
|---|---|
| $x_{\text{det}}$ | Determinista (Proyecto 1) |
| $x_{\text{sto}}$ | Esperado neutral (Proyecto 1, actualizado) |
| $x_{\text{risk}}$ | CVaR risk-averse (nuevo) |
| $x_{\text{EV}}$ | Solución del expected-value problem |

**Reporten para cada solución:**

1. Costo promedio (out-of-sample).
2. VaR₉₅ y CVaR₉₅ del costo.
3. Tasa de violación de restricciones (déficit, capacidad, servicio).
4. Comparación Mediocristan vs Extremistan: ¿cuál solución sobrevive mejor las colas?

## 6. Lecturas Obligatorias

Todos los grupos deben leer:

- **Rahmaniani et al.** (2017), *The Benders decomposition algorithm: A literature review*, EJOR.

Y *uno* de los dos siguientes:

- **Benders** (1962), *Partitioning procedures for solving mixed-variables programming problems*, Numerische Mathematik.
- **Van Slyke y Wets** (1969), *L-Shaped Linear Programs with Applications to Optimal Control and Stochastic Programming*, SIAM J. Applied Mathematics.

### Lecturas a elegir por grupo

Cada grupo elige al menos dos artículos adicionales de la siguiente lista (o de la revisión de Rahmaniani, 2017), de los cuáles debe hacer una síntesis escrita y presentar *uno* (relacionado con su proyecto) en clase ($\leq 7$ min):

| Referencia | Contribución clave |
|---|---|
| **Geoffrion** (1972) — *Generalized Benders Decomposition* | Extiende Benders a sub-problemas convexos (no sólo LP). El corte puede venir de dualidad convexa. |
| **Magnanti y Wong** (1981) — *Accelerating Benders Decomposition* | Formaliza cortes Pareto-óptimos (no dominados). No basta con agregar "algún" corte: hay cortes más fuertes. |
| **Ruszczyński** (1986) — *A regularized decomposition method* | Regularización cuadrática para estabilizar el master y evitar "rebotes". |
| **Birge y Louveaux** (1988) — *A multicut algorithm for two-stage stochastic LP* | Multi-cut: un corte por escenario en vez de uno agregado. Primera especialización tangible del esquema. |
| **Higle y Sen** (1991) — *Stochastic decomposition* | Versión muestral: Benders + SAA + estimación estadística del recourse. Conecta con ambientes de gran escala. |
| **Laporte y Louveaux** (1993) — *The integer L-shaped method* | Benders para stochastic integer programming. El esquema sobrevive el salto de LP a MIP. |
| **Zakeri, Philpott y Ryan** (2000) — *Inexact Cuts in Benders Decomposition* | No hace falta resolver el sub-problema exactamente; cortes inexactos conservan convergencia. |
| **Hooker y Ottosson** (2003) — *Logic-based Benders decomposition* | El corte puede venir de inferencia lógica, no de dualidad LP. Abre scheduling, constraint programming. |
| **Codato y Fischetti** (2006) — *Combinatorial Benders' Cuts for MILP* | Cortes combinatorios directos para evitar dependencias artificiales de big-M. |

## 7. Entregables

### A) Reporte ejecutivo (máx. 8 páginas) + presentación en clase (< 8 min)

1. **Recordatorio del problema** (breve, referenciando Proyecto 1).
2. **Formulación de dos etapas** limpia, con notación matemática clara.
3. **Comparación algorítmica:** tabla DE vs Naive vs B&C; gráficos de convergencia.
4. **Escalabilidad:** gráficos de estabilidad SAA y tiempo vs $N$.
5. **Métricas de decisión:** tabla de VSS, EVPI, CVaR; interpretación.
6. **Backtesting extendido:** comparación de soluciones out-of-sample.
7. **Síntesis de lecturas:** resumen de Rahmaniani + artículo clásico elegido + dos lecturas adicionales (máx. 1 página en total).
8. **Conclusiones:** ¿Cuándo vale la pena descomponer? ¿Cuándo vale la pena ser averso al riesgo?

### B) Notebook reproducible

El notebook debe:

- Importar y usar los módulos base del curso (como se han visto en los tutoriales de clase).
- Generar escenarios con semilla fija y ejecutar las tres estrategias de solución.
- Producir todas las tablas y figuras del reporte.
- Correr de principio a fin sin errores (probado por ustedes antes de entregar).

### C) Mini-bibliografía comentada

- Síntesis de la lectura obligatoria (Rahmaniani + Benders/Van Slyke).
- Síntesis de las 2 lecturas adicionales elegidas.
- Conexión explícita: ¿cómo se relaciona cada lectura con su implementación?

## 8. Notas Prácticas y Restricciones

- **Correctitud antes que escala.** Un modelo pequeño que da resultados verificables vale más que uno grande con bugs silenciosos. Verifiquen siempre contra el DE.
- **Reproducibilidad.** Semillas, dependencias, instrucciones claras. Si no corre, no se puede evaluar.
- **El riesgo debe importar.** Si $x_{\text{risk}}$ es idéntica a $x_{\text{sto}}$, expliquen por qué (o revisen su definición de pérdida). El CVaR debe "mover" las decisiones.
- **Gráficos de convergencia** son obligatorios. Un Benders sin plot de LB/UB es un Benders sin evidencia.
- **Comparen mundos.** Mediocristan vs Extremistan debe aparecer en la escalabilidad, en las métricas de decisión, y en el backtesting. Es el hilo conductor desde el Proyecto 1.